# Supervised Fine-Tuning v2 on Kaggle — NileChat-3B on the redacted WhatsApp data

The Kaggle version of [`sft_nilechat_v2.ipynb`](sft_nilechat_v2.ipynb). The data checks, the labels (loss on Ahmed's final reply only), the trainer, early stopping and the run summary are the **same code**; only the platform parts differ:

| Colab version | Kaggle version |
|---|---|
| Data from Google Drive | Data from a **private Kaggle dataset**, found by its `audit_report.json` |
| Token typed with `getpass` | Token from **Kaggle Secrets** (`HF_TOKEN`) |
| Scans for names with `private_names.txt` from Drive | The names file is **not uploaded**: the scan ran on your PC before upload. The audit check still refuses data built without names redaction |
| Merges the model on Drive | Saves `best_adapter` + `run_summary.json` as **one small zip** to download; `sft_v2_merge_adapter.ipynb` merges it on Colab |

Kaggle gives ~30 free GPU hours a week and sessions of up to 12 hours. With **Save Version → Save & Run All (Commit)** the notebook keeps running with the browser closed; this run needs ~4–5 hours.

## Before you start
1. Kaggle account with a **verified phone number** (needed for GPU and Internet).
2. **Create a private dataset** (Datasets → New Dataset, visibility **Private**) with three files from your redacted build: `train.jsonl`, `validation.jsonl`, `audit_report.json`. Do **not** upload `private_names.txt`.
3. **Add-ons → Secrets**: add `HF_TOKEN` = your Hugging Face token, and attach it to this notebook. The account must have accepted the NileChat-3B licence.
4. In the notebook: **Add Input** → your private dataset. **Settings → Accelerator: GPU T4 x2** (or P100). **Settings → Internet: On**.
5. **Save Version → Save & Run All (Commit)**. When it finishes, download `sft_v2_best_adapter.zip` from the version's **Output** tab.

## Then
Upload the zip to `MyDrive/ahmed_twin/` and run `sft_v2_merge_adapter.ipynb` on Colab (about 10 minutes) to create `model_merged_nilechat_v2`.

**Outputs can contain real messages** (Section 9 prints three replies): keep the notebook and its versions private.

## 1. Setup
### 1.1 Use one GPU

Unsloth trains on a single GPU. With "GPU T4 x2" the second card stays idle; hiding it keeps the trainer from trying to split work across both.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # before torch is imported

import torch
assert torch.cuda.is_available(), "No GPU -- Settings > Accelerator > GPU T4 x2 (or P100)"
print("GPU:", torch.cuda.get_device_name(0))

### 1.2 Install dependencies

In [ ]:
!pip install unsloth
!pip install --no-deps trl peft accelerate bitsandbytes  # as in v1; bitsandbytes provides the adamw_8bit optimizer

### 1.3 Imports and Hugging Face login

`unsloth` is imported first so its optimizations apply to `transformers` and `peft`. The token comes from Kaggle Secrets (`HF_TOKEN`) and is never written in the notebook. Library versions are recorded in the run summary.

In [ ]:
import unsloth  # must come before transformers / peft
from unsloth import FastLanguageModel

import gc
import hashlib
import json
import os
import re
import shutil
import statistics
import unicodedata
from collections import Counter

import datasets
import peft
import transformers
from datasets import Dataset
from huggingface_hub import login

from kaggle_secrets import UserSecretsClient

login(token=UserSecretsClient().get_secret("HF_TOKEN"))

LIBRARY_VERSIONS = {
    "unsloth": getattr(unsloth, "__version__", "unknown"),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "peft": peft.__version__,
    "datasets": datasets.__version__,
}
print(LIBRARY_VERSIONS)

## 2. Configuration

Everything you may want to change is in this one cell. `CONFIG` is saved with the run: resuming with a different `CONFIG` or different data is refused, so a run is never a mix of two settings.

In [ ]:
RUN_NAME = "sft_v2_redacted_final_reply_loss"
MODEL_NAME = "UBC-NLP/NileChat-3B"

INPUT_ROOT = "/kaggle/input"
OUTPUT_ROOT = "/kaggle/working"  # saved with the notebook version; download from its Output tab

# The private dataset is found by its audit_report.json, whatever it was named.
audit_files = [os.path.join(folder, "audit_report.json")
               for folder, _, files in os.walk(INPUT_ROOT) if "audit_report.json" in files]
assert len(audit_files) == 1, (
    f"Expected exactly one dataset with audit_report.json under {INPUT_ROOT}, found {len(audit_files)}. "
    "Add your private dataset with 'Add Input'."
)
DATA_DIR = os.path.dirname(audit_files[0])
TRAIN_PATH = f"{DATA_DIR}/train.jsonl"
VAL_PATH = f"{DATA_DIR}/validation.jsonl"
AUDIT_PATH = f"{DATA_DIR}/audit_report.json"
PRIVATE_NAMES_PATH = f"{DATA_DIR}/private_names.txt"  # deliberately not uploaded: see REQUIRE_NAME_CHECK

RUN_DIR = f"{OUTPUT_ROOT}/sft_runs/{RUN_NAME}"
CHECKPOINT_DIR = f"{RUN_DIR}/checkpoints"
BEST_ADAPTER_DIR = f"{RUN_DIR}/best_adapter"
RUN_CONFIG_PATH = f"{RUN_DIR}/run_config.json"
RUN_SUMMARY_PATH = f"{RUN_DIR}/run_summary.json"
DOWNLOAD_ZIP = f"{OUTPUT_ROOT}/sft_v2_best_adapter.zip"

CONFIG = {
    "max_seq_length": 2048,
    "loss_on": "final_reply_only",
    # LoRA: same as v1
    "lora_r": 32,
    "lora_alpha": 32,
    "lora_dropout": 0.0,
    "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    # Optimisation: same as v1 (effective batch 16)
    "learning_rate": 3e-4,
    "lr_scheduler_type": "linear",
    "warmup_steps": 10,
    "weight_decay": 0.01,
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 8,
    "per_device_eval_batch_size": 4,
    "num_train_epochs": 2,
    # New in v2: evaluate and save every 50 steps; stop after 4 evaluations (200 steps) without improvement
    "eval_steps": 50,
    "early_stopping_patience": 4,
    "seed": 0,
}

# The real names stay on your PC: the data was scanned there before upload (0 names), and Section 3.1
# still refuses any build made without the names file.
REQUIRE_NAME_CHECK = False
MAX_DROPPED_FRACTION = 0.01  # refuse to train if more than 1% of samples must be dropped
MIN_FREE_DISK_GB = 5         # checkpoints and the adapter; /kaggle/working holds about 20 GB

print("Data:", DATA_DIR)
print("Run folder:", RUN_DIR)

## 3. Data checks (before the GPU does any work)
### 3.1 The data must come from a redacted, validated build

`audit_report.json` is written by the data builder. It proves three things: the build validated its own output, it was built with the private names file, and these are the files it wrote (same record counts).

In [ ]:
def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def sha256_of(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


for path in (TRAIN_PATH, VAL_PATH, AUDIT_PATH):
    assert os.path.exists(path), f"Missing {path} -- upload the redacted build (see 'Before you start')."

with open(AUDIT_PATH, encoding="utf-8") as f:
    audit = json.load(f)

if audit.get("output_validation", {}).get("valid") is not True:
    raise ValueError("audit_report.json says the build did not pass its own validation.")
if audit.get("config", {}).get("private_names_count", 0) < 1:
    raise ValueError(
        "This data was built WITHOUT --redact-names-file (private_names_count is 0 or missing). "
        "Rebuild it with PRIVATE_NAMES_FILE set, then upload the new files."
    )

train_records = load_jsonl(TRAIN_PATH)
val_records = load_jsonl(VAL_PATH)
if (len(train_records), len(val_records)) != (audit["samples_train"], audit["samples_validation"]):
    raise ValueError(
        f"Record counts {len(train_records)}/{len(val_records)} do not match the audit "
        f"({audit['samples_train']}/{audit['samples_validation']}): the files and the audit come from different builds."
    )

for split, records in (("train", train_records), ("validation", val_records)):
    for i, record in enumerate(records):
        messages = record.get("messages")
        if not messages or messages[0].get("role") != "system" or messages[-1].get("role") != "assistant":
            raise ValueError(f"{split} record {i}: must start with 'system' and end with Ahmed's reply ('assistant').")

# The inference check must use the exact system message the model was trained with.
system_messages = {record["messages"][0]["content"] for record in train_records + val_records}
assert len(system_messages) == 1, f"Expected one system message in the data, found {len(system_messages)}."
SYSTEM_MESSAGE = system_messages.pop()

DATA_FINGERPRINT = {"train_sha256": sha256_of(TRAIN_PATH), "validation_sha256": sha256_of(VAL_PATH)}
print(f"train: {len(train_records)} | validation: {len(val_records)} | names in the redaction list: "
      f"{audit['config']['private_names_count']} | name redactions in the build: {audit['pii_redactions'].get('NAME', 0)}")

### 3.2 Scan for private names again

The same matching rules as `tools/private_names_lib.py` (whole words, with an attached و/ب/ل prefix). This prints **counts only**, never a name or a message. Any name found stops the notebook.

On Kaggle `private_names.txt` is not uploaded, so this scan is skipped: it already ran on your PC before the upload. Upload the file to the dataset only if you accept the real names being on Kaggle.

In [ ]:
ARABIC = "؀-ۿ"
NAME_PREFIXES = "وبل"  # not ف/ك: ف or ك + a short name often spells an everyday word


def load_private_names(path):
    names = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("#"):
                names.append(line)
    return sorted(set(names), key=len, reverse=True)  # longest first: full names before their parts


def compile_name_patterns(names):
    arabic = [n for n in names if re.search(f"[{ARABIC}]", n)]
    latin = [n for n in names if n not in arabic]
    ar = (re.compile(rf"(?<![{ARABIC}])(?P<pre>[{NAME_PREFIXES}])?(?P<name>{'|'.join(map(re.escape, arabic))})(?![{ARABIC}])")
          if arabic else None)
    la = re.compile(rf"(?<![A-Za-z])(?:{'|'.join(map(re.escape, latin))})(?![A-Za-z])", re.IGNORECASE) if latin else None
    return ar, la


def find_private_names(text, patterns):
    ar, la = patterns
    found = []
    if ar:
        found += [m.group("name") for m in ar.finditer(text)]
    if la:
        found += [m.group(0) for m in la.finditer(text)]
    return found


NAME_PATTERNS = None
if os.path.exists(PRIVATE_NAMES_PATH):
    NAME_PATTERNS = compile_name_patterns(load_private_names(PRIVATE_NAMES_PATH))
    hits = Counter()
    for split, records in (("train", train_records), ("validation", val_records)):
        for record in records:
            if any(find_private_names(m["content"], NAME_PATTERNS) for m in record["messages"][1:]):
                hits[split] += 1
    if hits:
        raise ValueError(f"Records containing a private name: {dict(hits)}. Rebuild the data with the names file.")
    print("Private-name scan: 0 records with a listed name.")
elif REQUIRE_NAME_CHECK:
    raise FileNotFoundError(f"{PRIVATE_NAMES_PATH} not found. Upload it, or set REQUIRE_NAME_CHECK = False.")
else:
    print("Private-name scan skipped here: it ran on your PC before upload, and 3.1 refused any unredacted build.")

## 4. Load the base model and tokenizer

16-bit, as in v1. `<|im_end|>` ends every turn in NileChat's chat template: it is the token the model must learn to write after a reply, and the token generation stops at.

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=CONFIG["max_seq_length"],
    load_in_4bit=False,
)

IM_END_ID = tokenizer.convert_tokens_to_ids("<|im_end|>")
assert IM_END_ID is not None and IM_END_ID != tokenizer.unk_token_id, "The tokenizer has no <|im_end|> token."
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"  # training pads on the right; labels on padding are ignored
print("Loaded:", MODEL_NAME, "| <|im_end|> id:", IM_END_ID, "| pad id:", tokenizer.pad_token_id)

## 5. Tokenize: loss on the final reply only

v1 gave the whole chat-template string to the trainer, so the loss counted the system message, Contact's messages and every reply in the context. Here each sample becomes:

```
input_ids:  <|im_start|>system ... <|im_start|>user ... <|im_start|>assistant\n في الطريق <|im_end|>
labels:     -100 -100 ... (everything before the final reply is ignored) ...   في الطريق <|im_end|>
```

`-100` means "do not count this token". Each of Ahmed's replies is the target of exactly one sample; it also appears in later samples' context, where it is not counted again.

Samples longer than `max_seq_length` are **dropped and counted**. Truncating them would cut the end of the sequence, which is the reply itself.

In [ ]:
def build_example(messages):
    # Prompt = everything up to and including '<|im_start|>assistant\n' of the final turn.
    prompt_text = tokenizer.apply_chat_template(messages[:-1], tokenize=False, add_generation_prompt=True)
    full_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    if not full_text.startswith(prompt_text):
        raise ValueError("chat template: the prompt is not a prefix of the full conversation")

    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    full_ids = tokenizer(full_text, add_special_tokens=False)["input_ids"]
    if full_ids[: len(prompt_ids)] != prompt_ids:
        raise ValueError("tokenization: the prompt tokens change when the reply is appended")

    reply_ids = full_ids[len(prompt_ids):]
    if IM_END_ID not in reply_ids:
        raise ValueError("the final reply has no <|im_end|>")
    reply_ids = reply_ids[: reply_ids.index(IM_END_ID) + 1]  # the reply and its <|im_end|>, nothing after

    input_ids = prompt_ids + reply_ids
    return {"input_ids": input_ids, "attention_mask": [1] * len(input_ids),
            "labels": [-100] * len(prompt_ids) + reply_ids}


def tokenize_split(records, split):
    examples, kept_records, dropped = [], [], Counter()
    for record in records:
        example = build_example(record["messages"])  # a format error is a bug: let it stop the notebook
        if len(example["input_ids"]) > CONFIG["max_seq_length"]:
            dropped["longer than max_seq_length"] += 1
            continue
        examples.append(example)
        kept_records.append(record)
    fraction = sum(dropped.values()) / len(records)
    if fraction > MAX_DROPPED_FRACTION:
        raise ValueError(f"{split}: {fraction:.1%} of samples would be dropped ({dict(dropped)}).")
    lengths = [len(e["input_ids"]) for e in examples]
    trained = [sum(label != -100 for label in e["labels"]) for e in examples]
    stats = {
        "samples": len(examples),
        "dropped": dict(dropped),
        "median_tokens": statistics.median(lengths),
        "max_tokens": max(lengths),
        "median_trained_tokens": statistics.median(trained),
        "trained_token_share": round(sum(trained) / sum(lengths), 4),
    }
    print(split, stats)
    return Dataset.from_list(examples), stats, kept_records


train_dataset, train_stats, train_kept = tokenize_split(train_records, "train")
val_dataset, val_stats, _ = tokenize_split(val_records, "validation")

### 5.1 Check the mask, on a made-up chat and on the real data

The made-up chat is printed in full so you can see exactly what is trained. The real data is checked silently: for every sample, the trained tokens must decode to exactly Ahmed's final reply plus `<|im_end|>`, compared after Unicode normalisation (NFC), because the tokenizer normalises text: on the real data, 11 replies held a letter in its decomposed form that comes back composed. Same text, different code points.

In [ ]:
def trained_text(example):
    return tokenizer.decode([t for t, l in zip(example["input_ids"], example["labels"]) if l != -100],
                            skip_special_tokens=False, clean_up_tokenization_spaces=False)


demo = [
    {"role": "system", "content": SYSTEM_MESSAGE},
    {"role": "user", "content": "انت فين؟"},
    {"role": "assistant", "content": "في الطريق"},
    {"role": "user", "content": "هتتأخر؟"},
    {"role": "assistant", "content": "ربع ساعة"},
]
demo_trained = trained_text(build_example(demo))
print("Made-up chat, trained part:", repr(demo_trained))
assert demo_trained == "ربع ساعة<|im_end|>", "Only the final reply and <|im_end|> may be trained."

# The tokenizer normalises Unicode (NFC): a letter stored in its decomposed form (e.g. ا + a combining
# madda instead of آ) comes back composed. Same text, different code points, so compare both sides as NFC.
nfc = lambda text: unicodedata.normalize("NFC", text)
mismatches = sum(nfc(trained_text(example)) != nfc(record["messages"][-1]["content"]) + "<|im_end|>"
                 for example, record in zip(train_dataset, train_kept))
assert mismatches == 0, f"{mismatches} samples train on something other than the final reply."
print(f"Real data: the trained part is exactly the final reply in all {len(train_kept)} training samples.")

## 6. LoRA adapters

Same as v1: rank 32 on every attention and MLP projection; the 3B base weights stay frozen.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    target_modules=CONFIG["target_modules"],
    random_state=CONFIG["seed"],
)
model.print_trainable_parameters()

## 7. Train
### 7.1 Same run, same settings: resume or refuse

The run's config and data fingerprints are saved in `run_config.json` the first time. On a re-run, a different config or different data is refused (use a new `RUN_NAME`). A finished run is not trained again.

In [ ]:
from transformers.trainer_utils import get_last_checkpoint

run_config = json.loads(json.dumps({"model": MODEL_NAME, "config": CONFIG, "data": DATA_FINGERPRINT}))
os.makedirs(RUN_DIR, exist_ok=True)
if os.path.exists(RUN_CONFIG_PATH):
    with open(RUN_CONFIG_PATH, encoding="utf-8") as f:
        saved_config = json.load(f)
    if saved_config != run_config:
        raise ValueError(f"{RUN_DIR} was started with a different config or different data. "
                         "Choose a new RUN_NAME instead of mixing two runs.")
else:
    with open(RUN_CONFIG_PATH, "w", encoding="utf-8") as f:
        json.dump(run_config, f, indent=2)

TRAINING_COMPLETE = os.path.exists(RUN_SUMMARY_PATH) and os.path.exists(BEST_ADAPTER_DIR)
LAST_CHECKPOINT = get_last_checkpoint(CHECKPOINT_DIR) if os.path.isdir(CHECKPOINT_DIR) else None
if TRAINING_COMPLETE:
    print("This run is already complete. Skipping to the results.")
elif LAST_CHECKPOINT:
    print("Resuming from", LAST_CHECKPOINT)
else:
    print("Starting a new run.")

### 7.2 Disk space

`/kaggle/working` holds about 20 GB. Checkpoints that fail to save would break resuming and the best-model tracking.

In [ ]:
free_gb = shutil.disk_usage(OUTPUT_ROOT).free / 1e9
print(f"Free space in {OUTPUT_ROOT}: {free_gb:.2f} GB")
assert free_gb > MIN_FREE_DISK_GB, f"Only {free_gb:.2f} GB free in {OUTPUT_ROOT}."

### 7.3 Trainer

A plain `transformers.Trainer` with our own labels (an `SFTTrainer` would rebuild labels from the text). `load_best_model_at_end` keeps the best checkpoint safe from `save_total_limit` and loads it when training ends; early stopping ends training after `early_stopping_patience` evaluations without improvement.

In [ ]:
from transformers import DataCollatorForSeq2Seq, EarlyStoppingCallback, Trainer, TrainingArguments

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    per_device_eval_batch_size=CONFIG["per_device_eval_batch_size"],
    learning_rate=CONFIG["learning_rate"],
    lr_scheduler_type=CONFIG["lr_scheduler_type"],
    warmup_steps=CONFIG["warmup_steps"],
    weight_decay=CONFIG["weight_decay"],
    num_train_epochs=CONFIG["num_train_epochs"],
    optim="adamw_8bit" if torch.cuda.is_available() else "adamw_torch",
    bf16=use_bf16,
    fp16=torch.cuda.is_available() and not use_bf16,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=CONFIG["eval_steps"],
    save_strategy="steps",
    save_steps=CONFIG["eval_steps"],  # must match eval_steps for load_best_model_at_end
    save_total_limit=2,               # the best checkpoint is never deleted by the rotation
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    remove_unused_columns=False,      # the dataset holds exactly input_ids, attention_mask and labels
    seed=CONFIG["seed"],
    data_seed=CONFIG["seed"],
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=DataCollatorForSeq2Seq(tokenizer, padding=True, pad_to_multiple_of=8, label_pad_token_id=-100),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=CONFIG["early_stopping_patience"])],
)

### 7.4 Train

Nothing to watch by hand: evaluation runs every 50 steps and training stops itself. Run it with **Save & Run All (Commit)** so it keeps going with the browser closed.

In [ ]:
if TRAINING_COMPLETE:
    print("Already complete -- not training again.")
else:
    model.config.use_cache = False  # not needed during training
    train_result = trainer.train(resume_from_checkpoint=LAST_CHECKPOINT)
    print(train_result)

## 8. Results

The best weights are already loaded (`load_best_model_at_end`). They are saved as `best_adapter`, which the next sections and any later session use. `run_summary.json` records everything needed to reproduce or audit the run.

In [ ]:
if TRAINING_COMPLETE:
    with open(RUN_SUMMARY_PATH, encoding="utf-8") as f:
        summary = json.load(f)
else:
    eval_history = [(h["step"], round(h["eval_loss"], 4)) for h in trainer.state.log_history if "eval_loss" in h]
    summary = {
        **run_config,
        "best_checkpoint": trainer.state.best_model_checkpoint,
        "best_eval_loss": trainer.state.best_metric,
        "stopped_at_step": trainer.state.global_step,
        "planned_steps": trainer.state.max_steps,
        "early_stopped": trainer.state.global_step < trainer.state.max_steps,
        "eval_history": eval_history,
        "train_data": train_stats,
        "validation_data": val_stats,
        "library_versions": LIBRARY_VERSIONS,
        "loss_note": "eval_loss counts only Ahmed's final reply and <|im_end|>; not comparable with v1 (all tokens).",
    }
    trainer.model.save_pretrained(BEST_ADAPTER_DIR)
    tokenizer.save_pretrained(BEST_ADAPTER_DIR)
    with open(RUN_SUMMARY_PATH, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)

print(f"Best eval loss {summary['best_eval_loss']:.4f} at {summary['best_checkpoint']}; "
      f"stopped at step {summary['stopped_at_step']} of {summary['planned_steps']} planned.")
for step, loss in summary["eval_history"]:
    print(f"  step {step:>5}: eval_loss {loss}")

## 9. Inference check

The training model is freed and the saved `best_adapter` is loaded, so this checks exactly what goes into the zip. Same decoding settings as v1: `temperature=0.4`, an explicit `<|im_end|>` stop. (Private names in the replies are counted on Colab, where the names file is available; here the count is skipped.)

In [ ]:
for name in ("trainer", "model"):
    globals().pop(name, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

best_model, best_tokenizer = FastLanguageModel.from_pretrained(
    model_name=BEST_ADAPTER_DIR, max_seq_length=CONFIG["max_seq_length"], load_in_4bit=False,
)
FastLanguageModel.for_inference(best_model)
EOS_ID = best_tokenizer.convert_tokens_to_ids("<|im_end|>")

test_prompts = ["احكيلي عن نفسك", "معلش هو انت فاضي دلوقتي؟", "عامل ايه النهاردة؟"]
replies = []
for prompt in test_prompts:
    text = best_tokenizer.apply_chat_template(
        [{"role": "system", "content": SYSTEM_MESSAGE}, {"role": "user", "content": prompt}],
        tokenize=False, add_generation_prompt=True,
    )
    encoded = best_tokenizer(text, return_tensors="pt", add_special_tokens=False).to(best_model.device)
    output = best_model.generate(**encoded, max_new_tokens=100, do_sample=True, temperature=0.4,
                                 repetition_penalty=1.1, no_repeat_ngram_size=4,
                                 eos_token_id=EOS_ID, pad_token_id=EOS_ID)
    reply = best_tokenizer.decode(output[0, encoded["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    replies.append(reply)
    print(prompt, "->", reply, "\n---")

if NAME_PATTERNS:
    print("Replies mentioning a private name:", sum(bool(find_private_names(r, NAME_PATTERNS)) for r in replies),
          "of", len(replies))

## 10. Package the result for download

The merged model would be ~6 GB, slow to download from Kaggle and upload to Drive. Instead, `best_adapter` (~0.2 GB) and `run_summary.json` go into one zip. Download it from the version's **Output** tab, upload it to `MyDrive/ahmed_twin/`, and run `sft_v2_merge_adapter.ipynb` on Colab.

In [ ]:
import zipfile

with zipfile.ZipFile(DOWNLOAD_ZIP, "w", zipfile.ZIP_DEFLATED) as archive:
    archive.write(RUN_SUMMARY_PATH, "run_summary.json")
    for folder, _, files in os.walk(BEST_ADAPTER_DIR):
        for name in files:
            path = os.path.join(folder, name)
            archive.write(path, os.path.join("best_adapter", os.path.relpath(path, BEST_ADAPTER_DIR)))

with zipfile.ZipFile(DOWNLOAD_ZIP) as archive:
    names = archive.namelist()
assert "run_summary.json" in names and "best_adapter/adapter_config.json" in names, "The zip is incomplete."
print(f"Ready to download: {DOWNLOAD_ZIP} ({os.path.getsize(DOWNLOAD_ZIP) / 1e6:.0f} MB, {len(names)} files)")